# NB04 - Active Learning
## Background
Active learning addresses the cost of annotation: instead of labeling data uniformly at random, query the examples where the model is most uncertain. This can achieve the same accuracy as passive learning with 10-30% of the labeled data. The key design decisions are: (1) the query strategy (which examples to label next), and (2) the query budget (how many examples per round).

Classical strategies: uncertainty sampling (max entropy), margin sampling (smallest decision margin), least confidence sampling. Core-set (Sener & Savarese, 2018) selects examples that are geometrically representative of the unlabeled pool. BADGE (Ash et al., 2020) combines gradient magnitude with diversity for batch active learning.

## What You'll Learn
- Uncertainty sampling: entropy, margin, and least confidence
- Core-set selection: farthest-point traversal in feature space
- Full active learning loop: query, label, retrain, evaluate
- Learning curves: labeled pool size vs accuracy

## Why This Matters
Active learning is standard practice in annotation-constrained domains: medical imaging, legal document review, and industrial defect detection. Understanding query strategies determines how efficiently your annotation budget translates to model performance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

# ── Dataset setup ─────────────────────────────────────────────────────────
np.random.seed(42)
n_total = 800
n_classes = 4
n_features = 6
n_per_class = n_total // n_classes

class_means = np.random.randn(n_classes, n_features) * 3
X_all, y_all = [], []
for c in range(n_classes):
    feats = class_means[c] + np.random.randn(n_per_class, n_features)
    X_all.append(feats); y_all.extend([c] * n_per_class)
X_all, y_all = np.vstack(X_all), np.array(y_all)

# Separate test set
test_idx = np.random.choice(n_total, 200, replace=False)
train_pool_idx = np.setdiff1d(np.arange(n_total), test_idx)
X_test, y_test = X_all[test_idx], y_all[test_idx]
X_pool, y_pool = X_all[train_pool_idx], y_all[train_pool_idx]  # unlabeled pool

print(f'Pool: {len(X_pool)} samples, Test: {len(X_test)} samples')

# ── Simple model for active learning ──────────────────────────────────────
class SoftmaxClassifier:
    def __init__(self, n_features, n_classes, lr=0.1, n_epochs=50, seed=0):
        np.random.seed(seed)
        self.W = np.random.randn(n_features, n_classes) * 0.1
        self.b = np.zeros(n_classes)
        self.lr = lr
        self.n_epochs = n_epochs

    def softmax(self, X):
        logits = X @ self.W + self.b
        logits -= logits.max(axis=1, keepdims=True)
        exp_l = np.exp(logits)
        return exp_l / exp_l.sum(axis=1, keepdims=True)

    def fit(self, X, y):
        for _ in range(self.n_epochs):
            probs = self.softmax(X)
            n = len(y)
            dlogits = probs.copy()
            dlogits[np.arange(n), y] -= 1
            dlogits /= n
            self.W -= self.lr * X.T @ dlogits
            self.b -= self.lr * dlogits.sum(axis=0)
        return self

    def predict_proba(self, X): return self.softmax(X)
    def predict(self, X): return np.argmax(self.softmax(X), axis=1)
    def accuracy(self, X, y): return np.mean(self.predict(X) == y)

print('Softmax classifier ready for active learning loop.')

In [ ]:
# ── Query strategies ──────────────────────────────────────────────────────
def entropy_sampling(probs: np.ndarray, n_query: int) -> np.ndarray:
    """Select examples with highest predictive entropy."""
    entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1)
    return np.argsort(entropy)[-n_query:]

def margin_sampling(probs: np.ndarray, n_query: int) -> np.ndarray:
    """Select examples with smallest margin between top-2 classes."""
    sorted_probs = np.sort(probs, axis=1)
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]
    return np.argsort(margin)[:n_query]

def least_confidence(probs: np.ndarray, n_query: int) -> np.ndarray:
    """Select examples with lowest max probability."""
    max_probs = probs.max(axis=1)
    return np.argsort(max_probs)[:n_query]

def coreset_selection(X_labeled: np.ndarray, X_pool: np.ndarray,
                       n_query: int) -> np.ndarray:
    """Farthest-point traversal: select points that maximize min-distance to labeled set."""
    if len(X_labeled) == 0:
        return np.random.choice(len(X_pool), n_query, replace=False)
    selected = []
    remaining = list(range(len(X_pool)))
    for _ in range(n_query):
        # Distance from each pool point to nearest labeled point
        min_dists = np.array([
            min(np.linalg.norm(X_pool[i] - xl) for xl in X_labeled)
            for i in remaining
        ])
        best = remaining[np.argmax(min_dists)]
        selected.append(best)
        X_labeled = np.vstack([X_labeled, X_pool[best]])
        remaining.remove(best)
    return np.array(selected)

def random_sampling(n_pool: int, n_query: int) -> np.ndarray:
    return np.random.choice(n_pool, n_query, replace=False)

print('Query strategies: Entropy, Margin, Least-Confidence, Core-set, Random')

In [ ]:
# ── Active learning loop ───────────────────────────────────────────────────
def active_learning_loop(strategy: str, n_init: int = 20, n_query: int = 20,
                          n_rounds: int = 15) -> List[Tuple[int, float]]:
    np.random.seed(42)
    pool_indices = list(range(len(X_pool)))
    labeled_indices = list(np.random.choice(pool_indices, n_init, replace=False))
    unlabeled_indices = [i for i in pool_indices if i not in labeled_indices]
    results = []

    for round_idx in range(n_rounds):
        X_labeled = X_pool[labeled_indices]
        y_labeled = y_pool[labeled_indices]

        model = SoftmaxClassifier(n_features, n_classes, n_epochs=100)
        model.fit(X_labeled, y_labeled)
        acc = model.accuracy(X_test, y_test)
        results.append((len(labeled_indices), acc))

        if not unlabeled_indices:
            break

        # Query
        X_unlabeled = X_pool[unlabeled_indices]
        probs = model.predict_proba(X_unlabeled)
        n_query_actual = min(n_query, len(unlabeled_indices))

        if strategy == 'entropy':
            query_local = entropy_sampling(probs, n_query_actual)
        elif strategy == 'margin':
            query_local = margin_sampling(probs, n_query_actual)
        elif strategy == 'least_conf':
            query_local = least_confidence(probs, n_query_actual)
        elif strategy == 'coreset':
            query_local = coreset_selection(X_labeled, X_unlabeled, n_query_actual)
        else:  # random
            query_local = random_sampling(len(unlabeled_indices), n_query_actual)

        new_indices = [unlabeled_indices[i] for i in query_local]
        labeled_indices.extend(new_indices)
        unlabeled_indices = [i for i in unlabeled_indices if i not in new_indices]

    return results

print('Running active learning loops...')
strategies = ['random', 'entropy', 'margin', 'least_conf']
all_results = {}
for strat in strategies:
    results = active_learning_loop(strat)
    all_results[strat] = results
    final_n, final_acc = results[-1]
    print(f'  {strat:12s}: final acc={final_acc:.3f} with {final_n} labeled samples')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'random': 'gray', 'entropy': 'blue', 'margin': 'green', 'least_conf': 'orange'}
for strat, results in all_results.items():
    ns = [r[0] for r in results]
    accs = [r[1] for r in results]
    ax.plot(ns, accs, '-o', markersize=5, label=strat, color=colors[strat], linewidth=1.5)

ax.set_title('Active Learning: Test Accuracy vs Labeled Pool Size')
ax.set_xlabel('Number of labeled samples'); ax.set_ylabel('Test accuracy')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s29_04_active_learning.png', dpi=100, bbox_inches='tight')
plt.show()

print('=== Active Learning Summary ===')
target_acc = 0.75
for strat, results in all_results.items():
    n_to_target = next((r[0] for r in results if r[1] >= target_acc), results[-1][0])
    print(f'  {strat:12s}: {n_to_target} samples to reach {target_acc:.0%} accuracy')